# DeepSeek

**DeepSeek** is a Chinese AI lab whose models are notable for two things: **frontier-class quality at a fraction of the cost**, and **open weights** (MIT-licensed) you can download and self-host. Its flagships are **DeepSeek-V3** (a huge Mixture-of-Experts chat/coding model) and **DeepSeek-R1** (a reasoning model trained largely by reinforcement learning that thinks in an explicit `<think>` chain before answering). The hosted API is **OpenAI-compatible**, so calling it is a `base_url` swap.

**Domain:** Proprietary Models & Coding AI  ·  **recommended addition**  ·  **runnable:** yes  ·  _live cells gate on `os.getenv("DEEPSEEK_API_KEY")`_

## 1. What & Why

DeepSeek matters for two reasons that reinforce each other: **economics** and **openness**.

- **Economics.** DeepSeek-V3 and R1 reach quality competitive with closed frontier models (GPT-4o / o1 class) while training and serving at *dramatically* lower cost. The published V3 training run was ~2.8M H800 GPU-hours — roughly an order of magnitude cheaper than comparable Western frontier runs — and the API is priced accordingly (single-digit dollars per million output tokens, with an aggressive cache discount). This is the headline: it broke the assumption that frontier reasoning required hyperscaler-scale budgets.
- **Openness.** The weights for V3, R1, and the Coder/Math families are published on Hugging Face under permissive (MIT) terms. You can run them yourself, fine-tune them, or use cheap **distilled** variants (R1's reasoning distilled into small Qwen/Llama backbones, down to 1.5B params that run on a laptop).

**The problem it solves.** You want strong reasoning or coding, but (a) closed-model pricing is too high for your volume, and/or (b) you need to self-host for privacy, air-gapped deployment, or to avoid vendor lock-in. DeepSeek gives you a credible frontier-quality option you can either *call cheaply* or *own outright*.

**When to reach for it**

- High-volume reasoning/coding where token cost dominates your bill.
- You need **open weights** — on-prem, fine-tuning, or no-external-API constraints.
- You want a strong reasoning model (R1) but want to see and store its chain-of-thought.

**When *not* to.** If you need the absolute top of a specific leaderboard, multimodal input (DeepSeek is text-first), the richest tool-use/agent ecosystem, or you have data-governance rules against sending data to a China-based API (self-hosting the open weights sidesteps the API but not the provenance question). For those, a Western frontier API (Claude, GPT, Gemini) may fit better.

## 2. Mental Model

Think of DeepSeek as **"frontier reasoning rebuilt to be cheap — by activating only a sliver of a giant model per token, and by learning to reason from reinforcement learning instead of expensive human demonstrations."**

```
   ┌──────────────────────── DeepSeek-V3 (the base) ────────────────────────┐
   │  671B TOTAL parameters, but only ~37B ACTIVATED per token (MoE)         │
   │                                                                         │
   │   token ─▶ router ─▶ picks ~8 of 256 experts (+ 1 shared) ─▶ output     │
   │                       │                                                 │
   │            you "rent" a 671B brain but pay to run a 37B one each step   │
   │   plus: Multi-head Latent Attention (MLA) shrinks the KV cache,         │
   │         FP8 training + Multi-Token Prediction cut the compute bill      │
   └─────────────────────────────────────────────────────────────────────────┘
                                   │  RL post-training (GRPO)
                                   ▼
   ┌──────────────────────── DeepSeek-R1 (the reasoner) ────────────────────┐
   │  learns to think via reinforcement learning, not human CoT examples     │
   │                                                                         │
   │  prompt ─▶  <think> long internal chain-of-thought … </think>           │
   │            final answer                                                  │
   │                                                                         │
   │  API splits this for you:  message.reasoning_content  +  message.content │
   └─────────────────────────────────────────────────────────────────────────┘
```

Three things to internalize:

1. **MoE = pay for what you activate.** V3 is 671B parameters but routes each token through only ~37B of them. You get a big model's knowledge at a small model's per-token compute — that's *why* it's cheap.
2. **R1's reasoning is *learned*, not *scripted*.** Instead of fine-tuning on human-written chains of thought, DeepSeek used large-scale RL (GRPO) with simple correctness rewards; the model *discovered* long deliberation on its own. R1-Zero was pure RL with no supervised warm-up at all.
3. **OpenAI-shaped API.** Same `chat/completions` body; `deepseek-chat` is V3, `deepseek-reasoner` is R1. The reasoner returns its chain-of-thought in a **separate** `reasoning_content` field — surface it or strip it, but never feed it back in the next turn.

## 3. Key Concepts

- **DeepSeek-V3** — the flagship general/coding model. A **Mixture-of-Experts** transformer: **671B total / ~37B activated** per token. This is what `deepseek-chat` serves.
- **DeepSeek-R1** — the reasoning model, post-trained from V3 with reinforcement learning. Emits an explicit chain-of-thought, then the answer. Served as `deepseek-reasoner`. **R1-Zero** is the pure-RL ablation (no SFT) that proved reasoning can emerge from RL alone.
- **Mixture-of-Experts (MoE) + MLA.** DeepSeekMoE uses many *fine-grained* experts plus always-on *shared* experts, with **auxiliary-loss-free** load balancing. **Multi-head Latent Attention (MLA)** compresses the KV cache into a low-rank latent, slashing inference memory. Together: cheaper to train and serve. (See `mixture-of-experts` notebook.)
- **GRPO (Group Relative Policy Optimization).** The RL algorithm behind R1: drops PPO's value network and instead normalizes rewards *within a group* of sampled answers — simpler and cheaper RL at scale.
- **Distilled models.** R1's reasoning distilled into small dense backbones — `DeepSeek-R1-Distill-Qwen-{1.5B,7B,14B,32B}` and `-Llama-{8B,70B}`. These run locally (Ollama/vLLM/transformers) and punch far above their size on math/code.
- **`reasoning_content`.** In the API, `deepseek-reasoner` returns the chain-of-thought in `message.reasoning_content`, separate from the final `message.content`. **Do not** echo `reasoning_content` back into the next request's messages — it's rejected/ignored.
- **Context caching (cache hit vs miss).** DeepSeek automatically caches prompt prefixes on disk. A **cache hit** input token is billed at a steep discount (often ~10×) vs a **cache miss**. Reusing a long system prompt / few-shot prefix is nearly free on input.
- **Open weights, MIT license.** V3, R1, and distills are downloadable from Hugging Face and usable commercially. The **hosted API** is OpenAI-compatible at `https://api.deepseek.com` with `Authorization: Bearer $DEEPSEEK_API_KEY`.

## 4. Setup

Two ways to use DeepSeek — pick based on whether you want *cheap hosted calls* or *self-hosted weights*.

```bash
# A) Hosted API (cheapest, fastest to try). Get a key:
#    https://platform.deepseek.com/  ->  API keys
export DEEPSEEK_API_KEY="sk-..."
pip install openai           # OpenAI-compatible; or use plain urllib (shown below)

# B) Self-host open weights. The full V3/R1 are huge (multi-GPU), but the
#    DISTILLED models run on modest hardware:
ollama run deepseek-r1:7b                     # easiest local path
# or via Hugging Face transformers / vLLM:
#   from transformers import AutoModelForCausalLM
#   AutoModelForCausalLM.from_pretrained("deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B")
```

Minimal hosted call (OpenAI SDK, gated below so the notebook always runs):

```python
from openai import OpenAI
client = OpenAI(api_key=os.environ["DEEPSEEK_API_KEY"],
                base_url="https://api.deepseek.com")
resp = client.chat.completions.create(
    model="deepseek-reasoner",                 # R1; use "deepseek-chat" for V3
    messages=[{"role": "user", "content": "Prove sqrt(2) is irrational."}],
)
print(resp.choices[0].message.reasoning_content)   # the <think> chain
print(resp.choices[0].message.content)             # the final answer
```

The cells below run top-to-bottom in a fresh kernel **without** the `openai` SDK or an API key: every network call is gated behind an `os.getenv` check, while the parsing and cost-estimation examples always execute.

In [1]:
# This notebook runs with or without an API key or the openai SDK.
# To run the live example:  export DEEPSEEK_API_KEY="sk-..."   (optionally pip install openai)
import os

api_key = os.getenv("DEEPSEEK_API_KEY")

try:
    import openai  # noqa: F401
    have_sdk = True
except ImportError:
    have_sdk = False

print("DEEPSEEK_API_KEY :", "set" if api_key else "(unset -- live API calls skipped)")
print("openai SDK       :", "installed" if have_sdk else "(not installed -- urllib fallback used)")
print()
print("Hosted models : deepseek-chat (V3, general/coding) | deepseek-reasoner (R1, reasoning)")
print("Endpoint      : https://api.deepseek.com  (OpenAI-compatible chat/completions)")
print("Open weights  : huggingface.co/deepseek-ai  (MIT; full + distilled variants)")

DEEPSEEK_API_KEY : (unset -- live API calls skipped)
openai SDK       : installed

Hosted models : deepseek-chat (V3, general/coding) | deepseek-reasoner (R1, reasoning)
Endpoint      : https://api.deepseek.com  (OpenAI-compatible chat/completions)
Open weights  : huggingface.co/deepseek-ai  (MIT; full + distilled variants)


## 5. Worked Examples

### Example 1 — Anatomy of a `deepseek-reasoner` (R1) response (no network)

R1's defining API trait is that the **chain-of-thought and the final answer come back in separate fields**: `message.reasoning_content` (the `<think>` deliberation) and `message.content` (the answer). Parsing a representative payload by hand puts the shape in muscle memory — and shows the multi-turn rule that bites everyone.

In [2]:
# A representative deepseek-reasoner (R1) chat/completions response (trimmed, OpenAI-shaped).
sample_response = {
    "id": "chatcmpl-r1-abc123",
    "model": "deepseek-reasoner",
    "choices": [{
        "index": 0,
        "finish_reason": "stop",
        "message": {
            "role": "assistant",
            # The chain-of-thought lives in its OWN field, not in `content`:
            "reasoning_content": (
                "Assume sqrt(2) = a/b in lowest terms. Then a^2 = 2 b^2, so a^2 is even, "
                "so a is even, a = 2k. Then 4k^2 = 2 b^2 => b^2 = 2k^2, so b is even too. "
                "But then a and b share factor 2 -- contradicts lowest terms."
            ),
            "content": "Therefore sqrt(2) is irrational. \u220e",
        },
    }],
    # R1 bills its thinking tokens; they show up in usage:
    "usage": {
        "prompt_tokens": 14,
        "completion_tokens": 96,          # includes the reasoning tokens
        "total_tokens": 110,
        "prompt_cache_hit_tokens": 0,
        "prompt_cache_miss_tokens": 14,
    },
}

msg = sample_response["choices"][0]["message"]
print("REASONING (message.reasoning_content):\n ", msg["reasoning_content"], "\n")
print("ANSWER (message.content):\n ", msg["content"], "\n")

u = sample_response["usage"]
print(f"tokens: {u['prompt_tokens']} in + {u['completion_tokens']} out "
      f"(reasoning counts toward output) = {u['total_tokens']}")

# THE multi-turn rule: keep `content` in history, DROP `reasoning_content`.
next_turn_history = [
    {"role": "user", "content": "Prove sqrt(2) is irrational."},
    {"role": "assistant", "content": msg["content"]},          # <-- answer only
    {"role": "user", "content": "Now do the same for sqrt(3)."},
]
assert all("reasoning_content" not in m for m in next_turn_history)
print("\nNext-turn history carries the ANSWER only -- reasoning_content is never fed back.")

REASONING (message.reasoning_content):
  Assume sqrt(2) = a/b in lowest terms. Then a^2 = 2 b^2, so a^2 is even, so a is even, a = 2k. Then 4k^2 = 2 b^2 => b^2 = 2k^2, so b is even too. But then a and b share factor 2 -- contradicts lowest terms. 

ANSWER (message.content):
  Therefore sqrt(2) is irrational. ∎ 

tokens: 14 in + 96 out (reasoning counts toward output) = 110

Next-turn history carries the ANSWER only -- reasoning_content is never fed back.


### Example 2 — Why it's cheap: MoE activation **and** the context-cache discount (no network)

Two levers drive DeepSeek's price. **(1) MoE:** you only run a fraction of the parameters per token. **(2) Context caching:** repeated prompt prefixes bill at a steep cache-hit discount. The snippet below makes both concrete. (Rates are illustrative — check the live pricing page; the *structure* is the lesson.)

In [3]:
# --- Lever 1: MoE only activates a sliver of the model per token ---
total_params_b     = 671      # DeepSeek-V3 total parameters (billions)
activated_params_b = 37       # activated per token (billions)
print(f"MoE: {activated_params_b}B activated of {total_params_b}B total "
      f"= {activated_params_b/total_params_b*100:.1f}% per token "
      f"(~{total_params_b/activated_params_b:.0f}x knowledge-to-compute ratio)")

# --- Lever 2: context caching. Illustrative rates, $ per 1M tokens ---
CACHE_HIT_IN  = 0.07     # cached prompt-prefix input tokens (big discount)
CACHE_MISS_IN = 0.27     # fresh input tokens
OUTPUT        = 1.10     # output tokens

def call_cost(cache_hit_in, cache_miss_in, out):
    return (cache_hit_in  / 1_000_000 * CACHE_HIT_IN
          + cache_miss_in / 1_000_000 * CACHE_MISS_IN
          + out           / 1_000_000 * OUTPUT)

# Scenario: a 2,000-token system+few-shot prefix reused across 1,000 requests,
# each adding a 50-token question and producing a 300-token answer.
N = 1_000
prefix, question, answer = 2_000, 50, 300

# Cold (no cache): every prefix token is a cache MISS every time.
cold = sum(call_cost(0, prefix + question, answer) for _ in range(N))
# Warm (cache): first request misses the prefix; the rest HIT it.
warm = call_cost(0, prefix + question, answer)                       # request 1: miss
warm += sum(call_cost(prefix, question, answer) for _ in range(N-1)) # 2..N: prefix hits

print(f"\n{N} requests, {prefix}-token reused prefix:")
print(f"  no cache : ${cold:.3f}")
print(f"  cached   : ${warm:.3f}   ({(1-warm/cold)*100:.0f}% cheaper on the reused prefix)")
print("\nLesson: put your stable system prompt / few-shot examples FIRST so they cache;")
print("only the small variable tail pays full price. MoE makes the model cheap to run;")
print("caching makes repeated context nearly free.")

MoE: 37B activated of 671B total = 5.5% per token (~18x knowledge-to-compute ratio)

1000 requests, 2000-token reused prefix:
  no cache : $0.884
  cached   : $0.484   (45% cheaper on the reused prefix)

Lesson: put your stable system prompt / few-shot examples FIRST so they cache;
only the small variable tail pays full price. MoE makes the model cheap to run;
caching makes repeated context nearly free.


### Example 3 — Call the live DeepSeek API (gated on `DEEPSEEK_API_KEY`)

The real thing. With `DEEPSEEK_API_KEY` set this sends a live request to `deepseek-reasoner` and prints **both** the reasoning chain and the answer; otherwise it prints the exact call shape so the notebook still executes cleanly. We use plain `urllib` (no SDK required) — the body is OpenAI-compatible, so swapping in the `openai` client is a one-liner.

In [4]:
# Live call -- gated so the notebook runs with or without a key.
import os, json, urllib.request, urllib.error

ENDPOINT = "https://api.deepseek.com/chat/completions"

def deepseek_ask(question, model="deepseek-reasoner"):
    body = json.dumps({
        "model": model,
        "messages": [{"role": "user", "content": question}],
        "max_tokens": 512,
        # note: deepseek-reasoner ignores temperature/top_p/penalties.
    }).encode()
    req = urllib.request.Request(
        ENDPOINT, data=body,
        headers={"Authorization": f"Bearer {os.environ['DEEPSEEK_API_KEY']}",
                 "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)

if api_key:
    try:
        data = deepseek_ask("What is 17 * 24? Think step by step.")
        msg = data["choices"][0]["message"]
        reasoning = (msg.get("reasoning_content") or "").strip()
        if reasoning:
            print("REASONING:\n ", reasoning[:500], "...\n")
        print("ANSWER:", msg["content"].strip())
        u = data.get("usage", {})
        print("\nusage:", {k: u[k] for k in u if "token" in k})
    except urllib.error.HTTPError as e:              # auth / quota / bad request
        print("Live call failed:", e.code, e.read().decode()[:200])
    except Exception as e:
        print("Live call failed:", type(e).__name__, e)
else:
    print("Skipping live API call (set DEEPSEEK_API_KEY to run).")
    print("Request body would be:")
    print("  POST https://api.deepseek.com/chat/completions")
    print('  {"model": "deepseek-reasoner", "messages": [{"role":"user","content":"..."}]}')
    print("  -> choices[0].message.reasoning_content  (the <think> chain)")
    print("  -> choices[0].message.content            (the final answer)")

Skipping live API call (set DEEPSEEK_API_KEY to run).
Request body would be:
  POST https://api.deepseek.com/chat/completions
  {"model": "deepseek-reasoner", "messages": [{"role":"user","content":"..."}]}
  -> choices[0].message.reasoning_content  (the <think> chain)
  -> choices[0].message.content            (the final answer)


## 6. Gotchas & Pitfalls

- **Don't feed `reasoning_content` back into the next turn.** Multi-turn chat with `deepseek-reasoner` must carry only the assistant's `content` (the answer) in history. Echoing the chain-of-thought back is rejected/ignored — and wastes tokens (see Example 1).
- **R1 ignores sampling and some features.** `deepseek-reasoner` disregards `temperature`, `top_p`, presence/frequency penalties, and (historically) doesn't support function calling or JSON mode. Don't rely on those knobs to steer it; use `deepseek-chat` (V3) when you need them.
- **You pay for thinking tokens.** R1's chain-of-thought counts as output tokens — a "short" answer can cost many tokens because the deliberation is long. Budget for it, and cap with `max_tokens`.
- **Cache only helps if the prefix is *stable and first*.** The disk cache keys on identical prompt prefixes. Put your system prompt / few-shot examples at the front and keep them byte-identical; a changing timestamp or per-request preamble busts the cache and you lose the ~10× discount.
- **`deepseek-chat` ≠ `deepseek-reasoner`.** `deepseek-chat` is V3 (fast, tool-capable, no visible reasoning); `deepseek-reasoner` is R1 (slow, thinks out loud, fewer features). Picking the wrong one means either missing reasoning or paying for it when you didn't need it.
- **Data residency / governance.** The hosted API runs on China-based infrastructure; prompts leave your jurisdiction. If that's a blocker, self-host the **open weights** instead of calling the API — but note the model provenance is unchanged.
- **The full models are huge.** V3/R1 are 671B-parameter MoE checkpoints needing multi-GPU (hundreds of GB). For local use, reach for the **distilled** variants (1.5B–70B), not the full weights — and remember a distill is a *small dense model taught to imitate R1*, not R1 itself.
- **Text-first.** DeepSeek's mainline models are not multimodal — no image/audio input like GPT-4o/Gemini. Don't reach for it for vision tasks.
- **Distill naming confusion.** `DeepSeek-R1-Distill-Qwen-7B` is a **Qwen** backbone fine-tuned on R1 outputs; `DeepSeek-R1` is the real 671B model. They behave very differently — don't benchmark one and attribute it to the other.

## 7. When to Use vs Alternatives

| You need… | Reach for | Why |
|---|---|---|
| **Cheap, strong reasoning at high volume** | **DeepSeek-R1** (`deepseek-reasoner`) | o1-class reasoning at a fraction of the price; chain-of-thought you can inspect. |
| **Cheap, strong general/coding chat** | **DeepSeek-V3** (`deepseek-chat`) | GPT-4o-class quality, MoE-cheap, tool-capable, with the context-cache discount. |
| **Open weights you can self-host / fine-tune** | **DeepSeek (HF, MIT)** or **Llama / Qwen / Mistral** | DeepSeek gives frontier-grade open weights; the others are smaller/easier to run. |
| **Reasoning on a laptop / edge** | **DeepSeek-R1-Distill (1.5B–32B)** via Ollama/vLLM | Distilled reasoning runs locally; no API, no data egress. |
| **Top-of-leaderboard frontier quality, multimodal, richest tooling** | **Claude / GPT / Gemini** | Western frontier APIs lead on agentic tool-use, vision, and ecosystem maturity. |
| **Strict data-governance / no China-based API** | **Self-hosted open weights** (DeepSeek or others) or a Western API | Keeps prompts in your jurisdiction; choose by your provenance policy. |
| **Web-grounded, cited answers** | **Perplexity Sonar** | DeepSeek answers from weights, not live search — see the `perplexity` notebook. |

**Honest trade-offs**

- **vs Claude/GPT/Gemini (closed frontier)** — DeepSeek wins decisively on **price** and on being **open**; the closed labs generally lead on multimodal input, agentic/tool-use polish, ecosystem, and data-governance comfort for Western enterprises. For raw text reasoning per dollar, DeepSeek is hard to beat.
- **vs other open models (Llama, Qwen, Mistral)** — DeepSeek's full V3/R1 are *stronger* but *much heavier* (671B MoE) than typical open models; Qwen/Llama are easier to actually self-host. Ironically R1's reasoning is most *usable* locally through its Qwen/Llama **distills**.
- **vs Perplexity / a RAG stack** — orthogonal. DeepSeek reasons from its weights; it doesn't search the live web or your private docs. Pair it with retrieval if you need grounding.

## 8. Resources

- **DeepSeek API docs (OpenAI-compatible)** — https://api-docs.deepseek.com/
- **Reasoning model (R1) API guide** — https://api-docs.deepseek.com/guides/reasoning_model
- **Context caching guide (cache hit/miss pricing)** — https://api-docs.deepseek.com/guides/kv_cache
- **Pricing** — https://api-docs.deepseek.com/quick_start/pricing
- **Open weights on Hugging Face** — https://huggingface.co/deepseek-ai
- **DeepSeek-R1 paper (RL for reasoning, GRPO)** — https://arxiv.org/abs/2501.12948
- **DeepSeek-V3 technical report (MoE, MLA, FP8, MTP)** — https://arxiv.org/abs/2412.19437
- **GitHub (model code + reports)** — https://github.com/deepseek-ai

**Related notebooks:** `mixture-of-experts` (the MoE/MLA architecture under the hood); `qwen`, `mistral`, `llama` (other open-weight models you'd self-host); `anthropic-claude-api`, `perplexity` (closed frontier / answer-engine alternatives).